In [5]:
import numpy as np
from numba import cuda
import math
import time

@cuda.jit
def matrix_subtract(A,B,C):

    row,col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row,col] = A[row,col] - B[row,col]


N = 1024

A = np.random.randint(0,100,(N,N)).astype(np.float32)
B = np.random.randint(0,100,(N,N)).astype(np.float32)

C = np.zeros_like(A)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C)

threads = (16,16)

blocks_x = math.ceil(N/16)
blocks_y = math.ceil(N/16)

start = time.time()

matrix_subtract[(blocks_x,blocks_y),threads](d_A,d_B,d_C)

cuda.synchronize()

gpu_time = time.time()-start

C = d_C.copy_to_host()

print("GPU Time =",gpu_time)
print("A->",A[:5,:5])
print("B->",B[:5,:5])
print("C->",C[:5,:5])

GPU Time = 0.07200288772583008
A-> [[98. 83. 96. 59. 38.]
 [44.  9. 89. 85. 24.]
 [62. 38.  1. 51. 15.]
 [ 0. 32. 22. 38. 62.]
 [38. 98. 39. 64. 53.]]
B-> [[26. 81. 25. 47. 24.]
 [50. 85. 26. 34. 46.]
 [54. 35. 46. 79. 38.]
 [68. 61. 61. 44. 12.]
 [60. 90. 66. 97. 30.]]
C-> [[ 72.   2.  71.  12.  14.]
 [ -6. -76.  63.  51. -22.]
 [  8.   3. -45. -28. -23.]
 [-68. -29. -39.  -6.  50.]
 [-22.   8. -27. -33.  23.]]
